In [ ]:
import os
import ast
import math
import glob
import random
import warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import soundfile as sf
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio.transforms as T
import timm

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')

class CFG:
    ROOT_DIR = '/kaggle/input/competitions/birdclef-2026'
    TRAIN_CSV = os.path.join(ROOT_DIR, 'train.csv')
    SAMPLE_SUB_CSV = os.path.join(ROOT_DIR, 'sample_submission.csv')
    TRAIN_AUDIO_DIR = os.path.join(ROOT_DIR, 'train_audio')
    
    NUM_FOLDS = 5
    BATCH_SIZE = 16
    EPOCHS = 20
    LEARNING_RATE = 1e-4
    NUM_WORKERS = 4
    
    SR = 32000
    DURATION_SECONDS = 5.0
    CHUNK_LENGTH = int(SR * DURATION_SECONDS)
    
    N_MELS = 128
    N_FFT = 2048
    HOP_LENGTH = 512
    FMIN = 20
    FMAX = 16000
    
    BACKBONE_NAME = 'vit_base_patch16_224'
    IMAGE_SIZE = (224, 224)
    
    GEO_LOSS_WEIGHT = 5.0



In [ ]:
def setup_data():
    train_df = pd.read_csv(CFG.TRAIN_CSV)
    sample_sub = pd.read_csv(CFG.SAMPLE_SUB_CSV)
    submission_labels = [c for c in sample_sub.columns if c != 'row_id']
    CFG.NUM_CLASSES = len(submission_labels)
    species_to_idx = {species: idx for idx, species in enumerate(submission_labels)}
    
    train_df['author'] = train_df['author'].fillna('unknown')
    train_df['stratify_label'] = train_df['primary_label']
    
    sgkf = StratifiedGroupKFold(n_splits=CFG.NUM_FOLDS)
    folds = list(sgkf.split(train_df, train_df['stratify_label'], groups=train_df['author']))
    return train_df, folds, submission_labels, species_to_idx

class BirdCLEFDataset(Dataset):
    def __init__(self, df, species_to_idx, is_train=True):
        self.df = df
        self.species_to_idx = species_to_idx
        self.is_train = is_train
        self.mel_transform = T.MelSpectrogram(
            sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX
        )
        self.time_masking = T.TimeMasking(time_mask_param=20)
        self.freq_masking = T.FrequencyMasking(freq_mask_param=20)

    def __len__(self):
        return len(self.df)

    def _load_audio(self, path):
        try:
            data, sample_rate = sf.read(path, dtype='float32')
            if data.ndim > 1: data = data.mean(axis=1)
            if sample_rate != CFG.SR:
                resampler = T.Resample(sample_rate, CFG.SR)
                # Robust 1D resampling safeguard
                data = resampler(torch.tensor(data).unsqueeze(0)).squeeze(0).numpy()
        except:
            data = np.zeros(CFG.CHUNK_LENGTH)
            
        waveform = torch.tensor(data)
        total_len = waveform.size(0)
        
        if total_len > CFG.CHUNK_LENGTH:
            if self.is_train:
                offset = random.randint(0, total_len - CFG.CHUNK_LENGTH)
                waveform = waveform[offset:offset + CFG.CHUNK_LENGTH]
            else:
                offset = (total_len - CFG.CHUNK_LENGTH) // 2
                waveform = waveform[offset:offset + CFG.CHUNK_LENGTH]
        elif total_len < CFG.CHUNK_LENGTH:
            waveform = F.pad(waveform, (0, CFG.CHUNK_LENGTH - total_len))
            
        return waveform

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(CFG.TRAIN_AUDIO_DIR, row['filename'])
        waveform = self._load_audio(audio_path)
        
        target = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
        primary = str(row['primary_label']).strip()
        if primary in self.species_to_idx:
            target[self.species_to_idx[primary]] = 1.0
            
        lat = float(row.get('latitude', -19.05))
        lon = float(row.get('longitude', -56.75))
        if np.isnan(lat): lat = -19.05
        if np.isnan(lon): lon = -56.75
        geo_target = torch.tensor(np.radians([lat, lon]), dtype=torch.float32)
        
        mel_spec = self.mel_transform(waveform)
        log_mel = torch.log(mel_spec + 1e-6)
        
        if self.is_train:
            log_mel = self.time_masking(log_mel)
            log_mel = self.freq_masking(log_mel)
            
        log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-6)
        
        image = log_mel.unsqueeze(0).unsqueeze(0)
        image = F.interpolate(image, size=CFG.IMAGE_SIZE, mode='bilinear', align_corners=False).squeeze(0).repeat(3, 1, 1)
        
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std
        
        return image, target, geo_target



In [ ]:
class DualHeadViT(nn.Module):
    def __init__(self, backbone_name=CFG.BACKBONE_NAME, num_classes=234):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, in_chans=3)
        in_features = self.backbone.head.in_features if hasattr(self.backbone, 'head') else 768
        self.backbone.reset_classifier(0)
        
        self.acoustic_head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )
        
        self.geo_head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 2)
        )
        
    def forward(self, x):
        features = self.backbone(x)
        return self.acoustic_head(features), self.geo_head(features)



In [ ]:
def train_knn_prior(train_df, submission_labels, species_to_idx):
    spatial_df = train_df.dropna(subset=['latitude', 'longitude']).copy()
    X_rad = np.radians(spatial_df[['latitude', 'longitude']].values)
    
    y_targets = np.zeros((len(spatial_df), len(submission_labels)), dtype=np.float32)
    for idx, row in enumerate(spatial_df.itertuples()):
        primary = str(row.primary_label).strip()
        if primary in species_to_idx:
            y_targets[idx, species_to_idx[primary]] = 1.0
            
    knn = KNeighborsRegressor(n_neighbors=15, weights='distance', metric='haversine')
    knn.fit(X_rad, y_targets)
    return knn

def train_one_epoch(model, loader, optimizer, crit_ac, crit_geo, device):
    model.train()
    total_loss = 0
    for images, targets, geo_targets in tqdm(loader, desc="Training Epoch"):
        images, targets, geo_targets = images.to(device), targets.to(device), geo_targets.to(device)
        
        optimizer.zero_grad()
        ac_out, geo_out = model(images)
        
        loss = crit_ac(ac_out, targets) + (CFG.GEO_LOSS_WEIGHT * crit_geo(geo_out, geo_targets))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def validate(model, loader, knn_model, device):
    model.eval()
    all_targets, all_blended_preds = [], []
    
    for images, targets, _ in tqdm(loader, desc="Validation Epoch"):
        images = images.to(device)
        ac_out, geo_out = model(images)
        
        probs_visual = torch.sigmoid(ac_out).cpu().numpy()
        
        predicted_coords = geo_out.cpu().numpy()
        probs_knn = knn_model.predict(predicted_coords)
        
        final_probs = np.zeros_like(probs_visual)
        for i in range(len(probs_visual)):
            if np.max(probs_visual[i]) > 0.85:
                final_probs[i] = probs_visual[i]
            else:
                final_probs[i] = probs_visual[i] * 0.7 + probs_knn[i] * 0.3
                
        all_targets.append(targets.numpy())
        all_blended_preds.append(final_probs)
        
    all_targets = np.concatenate(all_targets)
    all_blended_preds = np.concatenate(all_blended_preds)
    
    val_f1 = f1_score((all_targets > 0).astype(int), (all_blended_preds >= 0.5).astype(int), average='macro', zero_division=0)
    return val_f1



In [ ]:
def main():
    train_df, folds, submission_labels, species_to_idx = setup_data()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print("Training KNN Spatial Prior...")
    knn_model = train_knn_prior(train_df, submission_labels, species_to_idx)
    joblib.dump(knn_model, 'knn_spatial_prior.pkl')
    
    for fold in range(1):
        print(f"\n{'='*20} Starting FOLD {fold} {'='*20}")
        train_idx, val_idx = folds[fold]
        train_dataset = BirdCLEFDataset(train_df.iloc[train_idx], species_to_idx, is_train=True)
        val_dataset = BirdCLEFDataset(train_df.iloc[val_idx], species_to_idx, is_train=False)
        
        train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS)
        val_loader = DataLoader(val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS)
        
        model = DualHeadViT(num_classes=CFG.NUM_CLASSES).to(device)
        criterion_ac = nn.BCEWithLogitsLoss()
        criterion_geo = nn.MSELoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LEARNING_RATE)
        
        best_f1 = 0
        for epoch in range(1, CFG.EPOCHS + 1):
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion_ac, criterion_geo, device)
            val_f1 = validate(model, val_loader, knn_model, device)
            print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Blended F1: {val_f1:.4f}")
            
            if val_f1 > best_f1:
                best_f1 = val_f1
                torch.save(model.state_dict(), f'best_dual_head_vit_fold_{fold}.pth')
                print(f"   => Saved new best model with F1: {best_f1:.4f}!")

if __name__ == '__main__':
    main()

